[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/dec_stokes/dec_stokes.ipynb)

# Stokes Flow and Discrete Exterior Calculus

Stokes flow balances pressure against viscous resistance when inertia is negligible. This example finds the pressure needed to carry fluid through a channel around an obstacle. Every triangle passes on exactly as much fluid as it receives, and the solid walls bring the fluid to rest.

Integrated fluxes live on mesh edges as scalar cochains, while the circumcentric dual cells carry oriented area patches from the regressive product of vertex displacements. Incidence maps contain only signed connectivity, and diagonal Hodge maps scale by dual-to-primal cell ratios while supplying the complementary orientation. Summing dual-edge circulation around each dual polygon and dividing by its area gives the vertex vorticity bivector, whose edge differences drive viscous resistance.

In differential-form notation, the topological identity is $d_1d_0=0$, and the circumcentric Hodge entries are $|\star\sigma^k|/|\sigma^k|$. Unlike Whitney-form discretizations that reconstruct edge cochains into ambient multivector fields on triangles before applying a material law, canonical DEC keeps form degree on the mesh cell families and collapses the metric into diagonal Hodge maps between primal and dual cells. [Jagad et al.](https://arxiv.org/abs/2010.15520) give a pressure–velocity DEC formulation of incompressible flow.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga.algebras import VGA2D as ga
from numga.sparse import SparseExtensor, spdiag
from examples import instantiate
from examples.fluids.dec_stokes import render, scenarios
from examples.fluids.dec_stokes.numerical import solve_stokes
from examples.util.animation import save_figure, show_animation

mesh_module = instantiate("examples.util.mesh", ga)
Mesh, mv = mesh_module.Mesh, mesh_module.mv
Scalar = ga.gatype.scalar()                                # Scalar+
Bivector = ga.gatype.bivector()                            # Bivector+
Point = ga.gatype.antivector()                             # Antivector-
Patch = ga.gatype.antibivector()                           # Antibivector-

np.set_printoptions(precision=4, suppress=True)

## The channel

The obstacle leaves unequal gaps above and below it. A prescribed throughput divides between them, with the pressure drop determined by viscous resistance. Lengths are measured in half-channel widths, speeds in the mean inlet speed, and viscosity is one.

In [ ]:
n_divisions = 18
viscosity, mean_speed = 1.0, 1.0
height = 2.0
n_frames, duration_ms = 84, 40
mesh, vertices, inlet, outlet, walls = scenarios.channel(n_divisions)

save_figure(render.draw_mesh(mesh, vertices), "dec_stokes_mesh");

## Cochains and incidence

A cochain holds one signed integral per cell: vertex readings, edge integrals or face integrals. Form degree belongs to the cell family, tracked by the sparse operator's field dimensions over vertices, edges and faces. Incidence contains only signs: `d1(d0)` vanishes because each vertex occurs twice with opposite signs in a triangle's boundary.

In [ ]:
d0 = mesh.d[0] * Scalar                                    # (Scalar+ <- Scalar+)[edges <- vertices]
d1 = mesh.d[1] * Scalar                                    # (Scalar+ <- Scalar+)[faces <- edges]
d0_vorticity = mesh.d[0] * Bivector                        # (Bivector+ <- Bivector+)[edges <- vertices]

print("Vertex to edge:", d0.gatype, d0.shape)
print("Edge to face:  ", d1.gatype, d1.shape)

## Primal and dual cells

Each triangle has a dual vertex at its circumcentre, where the distances to its corners agree. Neighbouring circumcentres define the dual edge crossing their shared primal edge. Around a primal vertex these edges enclose a dual polygon. The primal mesh is blue and its dual is orange; the highlighted triangle and polygon show the two kinds of area.

In [ ]:
edges = mesh.d[0] * vertices                               # Point-[edges]
corners = mesh.incidence[2, 0] * vertices                  # [corners] Point-[faces]
a, b, c = corners
u, v = b - a, c - a
face_patches = (u & v) / 2                                 # Patch-[faces]
areas = face_patches.norm()                                # Scalar+[faces]
face_centres = corners.sum(axis=0) / 3                     # Point-[faces]

# The triangle's signed area patch turns the squared-length-weighted edge difference
# into the offset equidistant from all three corners.
weighted = v.norm_squared() * u - u.norm_squared() * v      # Point-[faces]
circumcentres = a + weighted.commutator(face_patches.inverse().dual()) / 4  # Point-[faces]

one_face = mv.scalar(np.ones((len(mesh.faces), 1))).field()  # Scalar+[faces]
midpoints = mesh.incidence[1, 0].mean(axis=0) * vertices   # Point-[edges]
boundary = ~mesh.d[1] * one_face                           # Scalar+[edges]
# Interior dual edges join adjacent circumcentres. At the boundary they
# finish at the primal edge's midpoint, closing the truncated dual cells.
# The dual edge crosses to the primal edge's right, matching positive flux.
dual_edges = boundary * midpoints - ~mesh.d[1] * circumcentres  # Point-[edges]

save_figure(render.draw_complex(mesh, vertices, circumcentres, dual_edges), "dec_stokes_complex");

## Diagonal Hodge maps

Joining each dual edge with its crossed primal edge gives the signed area of their kite as an intrinsic patch (`Patch`). Dividing by the squared primal edge length gives the dual-to-primal length ratio, and summing the quarter-kites around each vertex gives its dual polygon patch. Multiplying or dividing by these oriented patches maps primal readings directly to dual patch readings, while `inverse_star0` converts dual-polygon circulation into the vertex vorticity bivector.

In [ ]:
# Oriented dual/primal edge ratios and dual cell patches from the quarter-kites
# incident to each vertex.
edge_ratios = (dual_edges & edges) / edges.norm_squared()  # Patch-[edges]
dual_patches = ~mesh.incidence[1, 0].sum(axis=0) * ((dual_edges & edges) / 4)  # Patch-[vertices]
dual_areas = dual_patches.norm()                           # Scalar+[vertices]

star0 = spdiag(dual_patches * Scalar)                      # (Patch- <- Scalar+)[vertices <- vertices]
star1 = spdiag(edge_ratios * Scalar)                       # (Patch- <- Scalar+)[edges <- edges]
star2 = spdiag(face_patches.inverse() * Scalar)            # (Patch- <- Scalar+)[faces <- faces]
inverse_star0 = spdiag(Patch.dual_inverse() / dual_areas)  # (Bivector+ <- Patch-)[vertices <- vertices]

print("Primal to dual edge readings:", star1.gatype)
print("Dual circulation sum:        ", d0_vorticity.adjugate().gatype, d0_vorticity.adjugate().shape)

## Vorticity and viscous resistance

The dual of an edge flux is circulation along the crossing dual edge. Because a vorticity bivector pairs with a dual patch across `&`, `d0_vorticity.adjugate()` sums dual-edge circulations around each dual polygon, and `inverse_star0` divides by the polygon's area to give the vorticity bivector at each vertex. Vorticity differences across primal edges then pass through `star1.adjugate()` into the viscous edge response.

In [ ]:
curl = inverse_star0(d0_vorticity.adjugate()(star1))       # (Bivector+ <- Scalar+)[vertices <- edges]
viscous = viscosity * star1.adjugate()(d0_vorticity(curl)) # (Amount- <- Scalar+)[edges <- edges]

## Boundary flow and pressure

Pressure enforces zero net flux through every triangle. The inlet and outlet carry the same parabolic profile. On the solid walls, zero normal flux and zero tangential boundary circulation impose no slip. Fluxes and pressures are solved together, and the arbitrary pressure offset is fixed by its area-weighted mean.

In [ ]:
# The antiderivative of the prescribed inlet profile gives each boundary edge's flux.
ordinate = mv.y & vertices                                 # Scalar+[vertices]
inlet_potential = 1.5 * mean_speed * (ordinate - 4 * ordinate.squared() * ordinate / (3 * height**2))  # Scalar+[vertices]
ends = SparseExtensor.selection(mv.context, np.concatenate((inlet, outlet)), len(mesh.edges))
prescribed_flux = ~ends * (ends * d0(inlet_potential))     # Scalar+[edges]
fixed_edges = mesh.boundary_edges
boundary_circulation = dual_patches * 0                    # Patch-[vertices]

# Boundary segments close the dual polygons; their prescribed circulation
# contributes to vorticity and hence to viscous resistance.
boundary_vorticity = inverse_star0(boundary_circulation)   # Bivector+[vertices]
boundary_resistance = viscosity * star1.adjugate()(d0_vorticity(boundary_vorticity))  # Amount-[edges]

free_edges = np.setdiff1d(np.arange(len(mesh.edges)), fixed_edges)
embedding = ~SparseExtensor.selection(mv.context, free_edges, len(mesh.edges)) * Scalar  # (Scalar+ <- Scalar+)[edges <- free_edges]
stiffness = embedding.adjugate()(viscous(embedding))       # (Amount- <- Scalar+)[free_edges <- free_edges]
divergence = d1(embedding)                                 # (Scalar+ <- Scalar+)[faces <- free_edges]
force = -embedding.adjugate()(viscous(prescribed_flux) + boundary_resistance)  # Amount-[free_edges]
balance = -d1(prescribed_flux)                             # Scalar+[faces]
free_flux, pressure = solve_stokes(stiffness, divergence, force, balance)  # Scalar+[free_edges], Scalar+[faces]
flux = prescribed_flux + embedding(free_flux)              # Scalar+[edges]
pressure = pressure - (pressure * areas).sites.sum() / areas.sites.sum()  # Scalar+[faces]
vorticity = curl(flux) + boundary_vorticity                # Bivector+[vertices]

# The first moment of the outward flux gives the face's mean velocity.
outward_flux = mesh.boundary[2, 1] * flux                  # [corners] Scalar+[faces]
velocity = -((corners - face_centres) * outward_flux).sum(axis=0) / (2 * areas)  # Point-[faces]

# A streamfunction draws the conserved flux's paths on the native triangles.
streamfunction = d0.lstsq(flux)                            # Scalar+[vertices]
save_figure(render.draw_fields(mesh, vertices, pressure, vorticity, streamfunction), "dec_stokes_fields");

The upper panel shows pressure, falling from warm to cool colours across the obstacle. The lower panel shows the two senses of vorticity, concentrated where the flow shears past the walls and obstacle. The same streamlines cross both panels; their spacing records how the throughput divides.

In [ ]:
inlet_read = SparseExtensor.selection(mv.context, inlet, len(mesh.edges))
outlet_read = SparseExtensor.selection(mv.context, outlet, len(mesh.edges))
throughput = -(inlet_read * (boundary * flux)).sites.sum()
# The faces' pressures summed onto their edges: on a boundary edge, the pressure of the one face beside it.
edge_pressure = (~mesh.incidence[2, 1] * pressure).sum(axis=0)       # Scalar+[edges]
pressure_drop = (inlet_read * edge_pressure).sites.mean() - (outlet_read * edge_pressure).sites.mean()
print("Throughput per unit depth:", throughput.to_array())
print("Pressure drop near the ends:", pressure_drop.to_array())

## Flow paths

Tracers follow the steady flow, with their travel times set by the computed velocity. They follow the streamfunction contours on the triangles, including the constant contour along the obstacle boundary.

In [ ]:
show_animation(render.animate_flow(mesh, vertices, pressure, streamfunction, velocity, n_frames),
               "dec_stokes_flow", duration_ms, n_frames=n_frames)

## Reversing edge directions

Reversing an edge flips its signed flux reading and the incidences that refer to it. The corresponding dual edge reverses too, leaving its Hodge weight unchanged. Cell balances and the vorticity bivector at each vertex therefore stay invariant under any choice of edge directions.

In [ ]:
# Reverse every third edge, together with the primal and dual readings on it.
edge_signs = mv.scalar(np.where(np.arange(len(mesh.edges)) % 3 == 0, -1, 1)[:, None]).field()  # Scalar+[edges]
reorient = spdiag(edge_signs * Scalar)                     # (Scalar+ <- Scalar+)[edges <- edges]
reorient_vorticity = spdiag(edge_signs * Bivector)         # (Bivector+ <- Bivector+)[edges <- edges]
reversed_d0_vorticity = reorient_vorticity(d0_vorticity)   # (Bivector+ <- Bivector+)[edges <- vertices]
reversed_d1 = d1(reorient)                                 # (Scalar+ <- Scalar+)[faces <- edges]
reversed_star1 = reorient_vorticity.adjugate()(star1(reorient))  # (Patch- <- Scalar+)[edges <- edges]
reversed_flux = reorient(flux)                             # Scalar+[edges]
reversed_curl = inverse_star0(reversed_d0_vorticity.adjugate()(reversed_star1))  # (Bivector+ <- Scalar+)[vertices <- edges]

sample_edge = SparseExtensor.selection(mv.context, np.array([300]), len(mesh.edges))   # Scalar+[1 <- edges]
first_vertex = mesh.reading(np.array([0]))                 # Scalar+[1 <- vertices]
print("Sample edge flux before and after reversal:", (sample_edge * flux).batch().to_array(),
      (sample_edge * reversed_flux).batch().to_array())
print("Vertex 0 vorticity before and after:       ", (mv.xy.dual() & (first_vertex * vorticity)).batch().to_array(),
      (mv.xy.dual() & (first_vertex * reversed_curl(reversed_flux))).batch().to_array())

In [ ]:
# checks
from examples.fluids.dec_stokes import core

potential = (mv.x & vertices).squared() + 0.3 * (mv.y & vertices)
np.testing.assert_allclose(d1(d0(potential)).kernel, 0, atol=1e-9)
np.testing.assert_allclose(d1(flux).kernel, 0, atol=1e-9)
np.testing.assert_allclose(reversed_d1(reversed_flux).kernel, 0, atol=1e-9)
np.testing.assert_allclose((reversed_curl(reversed_flux) - vorticity).kernel, 0, atol=1e-9)
np.testing.assert_allclose((vorticity & d0_vorticity.adjugate()(star1(flux))).sites.sum().kernel,
                           (star1.adjugate()(d0_vorticity(vorticity)) & flux).sites.sum().kernel, atol=1e-9)

core_operators = core.Operators(mesh, vertices)
core_flux, core_pressure, core_vorticity, core_velocity = core.stokes(
    core_operators, viscosity, prescribed_flux, fixed_edges, boundary_circulation)
np.testing.assert_allclose(flux.kernel, core_flux.kernel, atol=1e-12)
np.testing.assert_allclose(pressure.kernel, core_pressure.kernel, atol=1e-12)
np.testing.assert_allclose(vorticity.kernel, core_vorticity.kernel, atol=1e-12)
np.testing.assert_allclose(velocity.kernel, core_velocity.kernel, atol=1e-12)